<a href="https://colab.research.google.com/github/Darshan8ak/ParcelFireRiskScore/blob/darshan/draft0_FinalFireRiskScore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FNSB Parcel Fire Risk Model

Scores every tax parcel in the Fairbanks North Star Borough on a 0-100 fire
hazard scale, so parcels can be ranked and compared on a given day.

$$\text{RISK} = 0.35\,\text{FUEL} + 0.15\,\text{EXPO} + 0.15\,\text{SLOPE} + 0.35\,\text{FWI}$$

## Inputs

| Component | Source | Weight |
|---|---|---|
| FUEL | `FNSB_veg` class, mapped to a 0-100 fuel score | 0.35 |
| EXPO | Mean fuel score in a 15-100 m annulus (ember reception) | 0.15 |
| SLOPE | 3DEP 10 m gradient, as % of the p95 slope | 0.15 |
| FWI | AKFF station Fire Weather Index, area-weighted to the parcel | 0.35 |

Also required: `FNSB_Tax_Parcels` for the parcel geometry, 13 AKFF station CSVs,
and optional `AK_Fire_Management_Options_FNSB` for the protection readout.

## Method

Every component is computed at 10 m on EPSG:3338 (Alaska Albers, equal-area) and
averaged to the parcel, so parcel values are area-weighted by construction. Caps
are taken from the observed distributions — p95 for slope, p98 for FWI — not set
by hand. FWI is interpolated by nearest-station area share, and stations with no
reading that day are dropped with the remaining fractions renormalised.

## Reading the score

Bands: 0-25 Low, 25-40 Moderate, 40-55 Elevated, 55-70 High, 70-100 Very high.

The score is a **hazard index, not a probability**. It says how readily a parcel
would carry fire, not how likely ignition is. It carries no structure, access or
suppression term, and vegetation is a static snapshot. See the final cell for
known limitations and citations.

## Running

Set `CURRENT_USER`, run Cells 1-8 once to build the caches, then Cell 9 onward
for the maps. Caches live on your own Drive and are keyed by a hash of their
inputs, so a changed setting rebuilds rather than silently reusing.

In [ ]:
# =============================================================================
# Cell 1 — Configuration, assets and shared helpers
# =============================================================================
# ========== !! EDIT THIS ONE LINE !! ==========
CURRENT_USER = "darshan"      # "darshan" or "emma"
# ==============================================

PROJECT_ID = {
    "darshan": "fireaid-507818",
    "emma":    "fireaid-508301",
}[CURRENT_USER]

import os, io, csv, glob, re, json, shutil, time, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import ee, geemap
from geopy.geocoders import Nominatim
from google.colab import drive, files   # files is for optional manual uploads

drive.mount("/content/drive")
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

print(f"running as {CURRENT_USER} against project {PROJECT_ID}")

# Nominatim needs a descriptive agent or it throttles; the address search uses it.
geocoder = Nominatim(user_agent="fnsb_fireaid_app")

# Project tree. WORK holds small JSON/parquet caches; EXP holds the two CSV
# exports. Both live on Drive so they survive a kernel restart.
ROOT = Path("/content/drive/MyDrive/FireAID")
WORK = ROOT / "work"
EXP  = ROOT / "GEE_exports"
RAW  = ROOT / "WeatherData"
for d in (WORK, EXP, RAW):
    d.mkdir(parents=True, exist_ok=True)

# Earth Engine assets, resolved through the project chosen above.
VEG_ASSET    = f"projects/{PROJECT_ID}/assets/FNSB_veg"
PARCEL_ASSET = f"projects/{PROJECT_ID}/assets/FNSB_Tax_Parcels"

# Fail here with a readable message rather than three cells later.
for _label, _asset in (("vegetation", VEG_ASSET), ("parcels", PARCEL_ASSET)):
    try:
        ee.data.getAsset(_asset)
    except Exception as exc:
        raise RuntimeError(
            f"{_label} asset not visible in {PROJECT_ID}: {_asset}\n"
            f"Copy it into that project, or check CURRENT_USER."
        ) from exc

# Fire management option polygons. Optional — try the local project, then the
# shared copy. If neither is visible the policy line simply reads blank.
MGMT_ASSET = None
for _cand in (f"projects/{PROJECT_ID}/assets/AK_Fire_Management_Options_FNSB",
              "projects/fireaid-508301/assets/AK_Fire_Management_Options_FNSB"):
    try:
        ee.data.getAsset(_cand)
        MGMT_ASSET = _cand
        break
    except Exception:
        continue
if MGMT_ASSET is None:
    print("fire management options asset not visible — policy line will be blank")


# Everything is computed and exported on this projection and pixel size. EPSG:3338
# is Alaska Albers, equal-area, so parcel pixel counts are proportional to parcel
# area and the mean reducers are not biased by latitude. 10 m matches the DEM
# native resolution so the slope cap transfers to the applied slope without a
# resolution mismatch.
WORK_CRS, WORK_SCALE = "EPSG:3338", 10

def sig(*parts):
    """Short stable hash of whatever a cache file depends on."""
    return hashlib.md5("|".join(str(p) for p in parts).encode()).hexdigest()[:8]

def run_export(collection, description, selectors):
    """Start a table export, wait for it with a bounded poll, return the file.

    Every cached export goes through here, so the timeout cap, the completion
    assertion and the Drive-visibility wait exist once instead of being
    copy-pasted into each cell.
    """
    task = ee.batch.Export.table.toDrive(
        collection=collection, description=description, folder="GEE_exports",
        fileNamePrefix=description, fileFormat="CSV", selectors=selectors)
    task.start()
    print("task:", task.id)

    # Poll until the task reaches a terminal state. Without the cap this loop
    # would hang forever on a task stuck in READY.
    for _ in range(90):
        st = task.status()["state"]
        if st in ("COMPLETED", "FAILED", "CANCELLED"):
            break
        print("  ", st); time.sleep(20)
    assert task.status()["state"] == "COMPLETED", task.status().get("error_message")

    hits = []
    for _ in range(40):
        hits = glob.glob(f"/content/drive/MyDrive/**/{description}*.csv",
                         recursive=True)
        if hits: break
        time.sleep(10)
    assert hits, f"{description} not visible in Drive"
    # Newest match wins. The description is keyed, so any match is a file for
    # this exact configuration.
    return Path(max(hits, key=os.path.getmtime)).resolve()

W_FUEL, W_EXPO, W_SLOPE, W_FWI = 0.35, 0.15, 0.15, 0.35
assert abs(sum([W_FUEL, W_EXPO, W_SLOPE, W_FWI]) - 1) < 1e-9

# Every percentile a cap is allowed to be drawn from
PCTS = [50, 75, 90, 95, 98, 99, 99.5, 99.9]

SLOPE_CAP_PCT = 95     # percentile of slope that scores 100 points
FWI_CAP_PCT   = 98     # percentile of FWI that scores 100 points
assert SLOPE_CAP_PCT in PCTS, f"SLOPE_CAP_PCT {SLOPE_CAP_PCT} not in {PCTS}"
assert FWI_CAP_PCT in PCTS,   f"FWI_CAP_PCT {FWI_CAP_PCT} not in {PCTS}"
SLOPE_SAMPLES = 150_000
SLOPE_SEED    = 42

STIDS = ["CPKA2", "CHTA2", "PAFA", "FRBA2", "BKSA2",
         "PAEI", "LXRA2", "STUA2", "AGLA2", "SLRA2", "BDRA2", "BONA2", "BENA2"]
FCOLS = [f"F{i:02d}" for i in range(1, len(STIDS) + 1)]

# (lat, lon) in decimal degrees.
STATION_META = {
    "PAFA":  (64.80389, -147.87611), "FRBA2": (64.84869, -147.57446),
    "PAEI":  (64.65000, -147.10000), "LXRA2": (64.65417, -146.92606),
    "CHTA2": (65.01933, -148.58333), "AGLA2": (65.02000, -146.22806),
    "CPKA2": (65.19183, -147.50017), "SLRA2": (64.59392, -146.14583),
    "STUA2": (64.71828, -146.50183), "BKSA2": (64.38583, -147.69528),
    "BDRA2": (64.64451, -146.44540), "BONA2": (64.74296, -148.31635),
    "BENA2": (65.28333, -143.06667),
}

# Ember exposure radii
R_OUT, R_IN = 100, 15

# Records before this year are dropped because the FWI numbers look different
# and some stations didnt have data before this year
YEAR_MIN = 2015
SEASON   = ((4, 1), (10, 31))

# Cache names carry a hash of their inputs so a changed setting cannot reuse a
# stale file.
FWI_PQ     = WORK / f"station_fwi_daily_{sig(YEAR_MIN, SEASON, '|'.join(STIDS))}.parquet"
DEM_ID = "USGS/3DEP/10m_collection"
SLOPE_JSON = WORK / f"slope_pct_{sig(DEM_ID, VEG_ASSET, SLOPE_SAMPLES, SLOPE_SEED)}.json"
CAPS_JSON  = WORK / f"risk_caps_{sig(SLOPE_CAP_PCT, FWI_CAP_PCT, SLOPE_SAMPLES, SLOPE_SEED, '|'.join(STIDS))}.json"

# AKFF daily download endpoint, used by the live fetch in Cell 3.
AKFF_URL = "https://akff.mesowest.org/download/"

# AKFF rejects a bare request, so the headers spoof a browser session on the
# download page. Without the Referer and Origin the POST returns an error page.
AKFF_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Content-Type": "application/x-www-form-urlencoded",
    "Referer": "https://akff.mesowest.org/download/",
    "Origin": "https://akff.mesowest.org",
}

In [ ]:
# =============================================================================
# Cell 2 — Vegetation classes, fuel weights, and the rationale
# =============================================================================
# The 18 classes in the FNSB vegetation layer are scored 0-100 for fuel loading,
# fuel continuity and expected fire behaviour in interior Alaska boreal forest.
#
# GOVERNING PRINCIPLE: conifer-dominated stands carry higher and more continuous
# fuel and support deeper duff burning than deciduous stands, which hold more
# moisture and are seasonally rather than chronically flammable. Poorly drained
# wetland and alder resist deep burning.
#
# TIERS (weight):
#   100  Closed conifer (3, 4, 13) — continuous crown fuel over a deep organic
#        layer; the most hazardous arrangement in the boreal. [1][3]
#    80  Open conifer, woodland, grass/shrub (1, 2, 5, 17) — the same fuel type
#        at lower continuity, OR flashy fine fuel that cures early and carries
#        fire fast. [1][3][4][6]
#    65  Conifer-deciduous mixes (11, 12) — the deciduous component breaks up
#        continuity and raises live moisture, but the conifer component retains
#        crown and ladder fuel. Scored between the two rather than with either.
#        [1][5]
#    45  Pure deciduous (6-10) — high live moisture, no thick duff; resistant
#        but not immune, and flammable once cured. [1][4]
#    20  Alder, residual non-fuel, wetland (14, 15, 16) — poor drainage or
#        negligible load. [1][4]
#     0  Water and no-data (0) — also drives the NO_VEG flag in Cell 8.
#
# SOURCES
#   [1] Kasischke, Verbyla et al. (2010), "Alaska's changing fire regime —
#       implications for the vulnerability of its boreal forests", Can. J. For.
#       Res. 40: 1313-1324. Fire regime synthesis; conifer-vs-deciduous fuel
#       gradient; poorly drained sites resist deep burning.
#   [2] Chapin, Oswood, Van Cleve, Viereck, Verbyla (eds.) (2006), Alaska's
#       Changing Boreal Forest, OUP. Post-fire succession; spruce regeneration.
#   [3] Grzesik, Hollingsworth, Ruess, Turetsky (2022), "Fuel loads and plant
#       traits as community-level predictors...", JGR Biogeosciences 127(3),
#       e2021JG006696. Black spruce carries inherently higher burning potential.
#   [4] USDA Climate Hubs — interior AK fire severity and the conifer-to-
#       deciduous shift; deciduous moisture demand, no thick duff.
#   [5] Johnstone, Chapin, Hollingsworth et al. (2010), "Changes in fire regime
#       break the legacy lock on successional trajectories...", Glob. Change
#       Biol. 16(4): 1281-1295. Stand composition drives fire behaviour.
#   [6] Dynamic World (Brown et al., 2022), Google/WRI. Grass and shrub/scrub
#       used to resolve flashy fine fuel the primary classification misses.
#
# CAVEAT: class 15 ("Others") is a residual — bare ground, alpine, impervious —
#   remaining AFTER grass/shrub was extracted into class 17 via Dynamic World.
#   Scored 20 as genuine non-fuel; class 17 is scored 80 for flashy fine fuel.
#
# One dict is the single source of truth: (display name, hex colour, weight).
# The three lookups downstream are derived from it, so a colour or weight can
# never drift out of step with the legend.
# =============================================================================
VEGETATION = {
    0:  ("No Data / Water", "#000000", 0),
    1:  ("Open black spruce forest", "#004D40", 80),
    2:  ("Black spruce woodland", "#1E88E5", 80),
    3:  ("Closed black spruce forest", "#003300", 100),
    4:  ("Closed white spruce forest", "#3949AB", 100),
    5:  ("White spruce woodland", "#5E35B1", 80),
    6:  ("Open paper birch forest", "#689F38", 45),
    7:  ("Open quaking aspen forest", "#43A047", 45),
    8:  ("Open balsam poplar forest", "#2E7D32", 45),
    9:  ("Closed paper birch - quaking aspen forest", "#1565C0", 45),
    10: ("Open paper birch - quaking aspen forest", "#0288D1", 45),
    11: ("Open spruce - balsam poplar forest", "#6D4C41", 65),
    12: ("Open spruce - paper birch forest", "#8D6E63", 65),
    13: ("Closed spruce - paper birch - quaking aspen forest", "#FFA000", 100),
    14: ("Closed tall alder", "#5D4037", 20),
    15: ("Others (bare / impervious / non-fuel)", "#424242", 20),
    16: ("Wetlands", "#01579B", 20),
    17: ("Grass / Shrub", "#7CB342", 80),
}

# Derived lookups. VEG_PALETTE is positional, so VEGETATION must stay in class
# order — which it is, written 0-17.
VEGETATION_LEGEND     = {k: (n, c) for k, (n, c, w) in VEGETATION.items()}
VEGETATION_TO_WEIGHT  = {k: w for k, (n, c, w) in VEGETATION.items()}
VEG_PALETTE           = [c for _, c, _ in VEGETATION.values()]
ALL_CLASS_VALUES      = list(VEGETATION)
ALL_CLASS_COLORS      = VEG_PALETTE

In [ ]:
# =============================================================================
# Cell 2b — (optional) upload the station CSVs from this machine
# =============================================================================
# Only needed once per machine, and only if RAW is empty. Saves into the Drive
# folder Cell 3 reads from, so the location is handled for you. Multi-select the
# whole batch in the picker; files for stations outside STIDS are saved and
# ignored.
uploaded = files.upload()

for name, blob in uploaded.items():
    base = Path(name).name
    # Tolerate a browser " (1)" duplicate suffix by matching only the prefix.
    m = re.match(r"akff_download([A-Za-z0-9]+).*\.csv$", base, re.I)
    assert m, f"unexpected filename: {base}"
    stid = m.group(1).upper()
    # Canonical name, so a suffixed duplicate overwrites instead of sitting
    # alongside the real file.
    (RAW / f"akff_download{stid}.csv").write_bytes(blob)
    print("saved", f"akff_download{stid}.csv",
          "" if stid in STIDS else "  (not in STIDS, ignored)")

# Every station in STIDS must now be on disk before Cell 3 can build the table.
_have = set()
for p in RAW.glob("akff_download*.csv"):
    m = re.search(r"akff_download([A-Za-z0-9]+).*\.csv$", p.name, re.I)
    if m:
        _have.add(m.group(1).upper())
_missing = sorted(set(STIDS) - _have)
assert not _missing, f"still missing: {_missing}"

In [ ]:
# =============================================================================
# Cell 3 — Load station FWI series, and live-fetch dates outside the cache
# =============================================================================
# Reads the AKFF station CSVs, cuts to the comparable period and the fire season,
# and caches a wide date x station table. Runs before calibration because the FWI
# cap is drawn from this data.
#
# Forward-fill only, five days, because FWI is a path-dependent index: a missing
# day is better bridged than reset. The limit stops a dead station being carried
# across a whole season.
#
# ensure_fwi() extends the table on demand for a date past the cached range. It
# posts to the same AKFF endpoint the CSVs came from and writes the result into
# the parquet, so a fetched date is cached like any other and never re-requested.
# =============================================================================


def canon(c):
    c = str(c).strip().lower()
    if c in ("stid", "station", "stationid", "stn", "code"): return "stid"
    if c in ("dattim", "valid", "datetime", "date", "time"): return "dattim"
    return c

def load_any(p):
    """Read one station CSV, tolerating comment lines and odd delimiters."""
    with open(p, errors="replace", newline="") as fh:
        lines = [l for l in fh.read().splitlines()
                 if l.strip() and not l.startswith("!")]
    if len(lines) < 2:
        return pd.DataFrame()
    # Sniff the delimiter from the first two rows rather than assuming a comma.
    try:
        sep = csv.Sniffer().sniff(lines[0] + "\n" + lines[1], delimiters=",\t;|").delimiter
    except Exception:
        sep = ","
    d = pd.read_csv(io.StringIO("\n".join(lines)), dtype=str, sep=sep,
                    on_bad_lines="skip", engine="python")
    d.columns = [canon(c) for c in d.columns]
    return d.loc[:, ~d.columns.duplicated()]


# Reuse the cached wide table when it exists. The name is keyed on the period and
# the station set, so it cannot be stale for a different configuration.
if FWI_PQ.exists():
    FWI = pd.read_parquet(FWI_PQ).sort_index().reindex(columns=STIDS)
else:
    frames = []
    for p in sorted(RAW.glob("akff_download*.csv")):
        # Tolerate a browser " (1)" duplicate suffix; skip anything unparseable.
        _m = re.search(r"akff_download([A-Za-z0-9]+).*\.csv$", p.name, re.I)
        if not _m:
            continue
        stid = _m.group(1).upper()
        if stid not in STIDS:
            continue                      # ignore stations outside the chosen set
        d = load_any(p)
        if d.empty or "fwi" not in d.columns or "dattim" not in d.columns:
            continue
        d["stid"] = stid
        frames.append(d)

    assert frames, f"no akff_download*.csv found in {RAW}"

    df = pd.concat(frames, ignore_index=True)

    # Every station in the set must have produced at least one row. A missing
    # file would otherwise surface much later as a blank column.
    _missing = sorted(set(STIDS) - set(df["stid"]))
    assert not _missing, f"no CSV rows for {_missing}"

    # dattim is a timestamp string; strip to its first eight digits and parse as
    # YYYYMMDD. Anything unparseable becomes NaT and is dropped below.
    df["date"] = pd.to_datetime(
        df["dattim"].astype(str).str.replace(r"\D", "", regex=True).str[:8],
        format="%Y%m%d", errors="coerce")
    df["fwi"] = pd.to_numeric(df["fwi"], errors="coerce")
    df = (df.dropna(subset=["date", "fwi"])
            .drop_duplicates(["stid", "date"], keep="last")
            .sort_values(["stid", "date"]))

    df = df[df.date.dt.year >= YEAR_MIN]
    df = df[df.date.dt.strftime("%m-%d").between("%02d-%02d" % SEASON[0],
                                                 "%02d-%02d" % SEASON[1])]

    # Pivot to date x station, force the column order, and bridge gaps of up to
    # five days. reindex also creates an all-NaN column for any station that had
    # no rows at all.
    FWI = (df.pivot(index="date", columns="stid", values="fwi")
             .sort_index().reindex(columns=STIDS).ffill(limit=5))
    FWI.to_parquet(FWI_PQ)

assert not FWI.dropna(how="all").empty, "FWI table has no usable rows"
print(FWI.shape, FWI.index.min().date(), FWI.index.max().date())


def _station_day_fwi(stid, day):
    """One station's FWI for one day from the live AKFF endpoint, or None.

    The daily payload returns hourly rows; the last non-null FWI is taken to
    match the convention used when the CSVs were loaded (keep="last"), so a
    fetched day is directly comparable to a cached one.
    """
    ymd = day.strftime("%Y%m%d")
    yr, mo, dy = day.strftime("%Y"), day.strftime("%m"), str(day.day)
    payload = [("dtype", "daily"), ("stid", stid),
               ("sy-d", yr), ("sm-d", mo), ("sd-d", dy),
               ("ey-d", yr), ("em-d", mo), ("ed-d", dy),
               ("daily-startdate", ymd), ("daily-enddate", ymd)]
    try:
        r = requests.post(AKFF_URL, headers=AKFF_HEADERS, data=payload, timeout=15)
        if r.status_code != 200:
            return None
        lines = [l for l in r.text.splitlines()
                 if l.strip() and not l.startswith(("!", "#"))]
        if len(lines) < 2:
            return None
        try:
            sep = csv.Sniffer().sniff(lines[0] + "\n" + lines[1],
                                      delimiters=",\t;|").delimiter
        except Exception:
            sep = ","
        d = pd.read_csv(io.StringIO("\n".join(lines)), dtype=str, sep=sep,
                        on_bad_lines="skip", engine="python")
        d.columns = [c.strip().upper() for c in d.columns]
        if "FWI" not in d.columns:
            return None
        v = pd.to_numeric(d["FWI"], errors="coerce").dropna()
        return float(v.iloc[-1]) if len(v) else None
    except Exception:
        return None


def ensure_fwi(day):
    """Make sure `day` is in FWI, fetching and caching it if it is not.

    Returns True when the date has at least one station reading.
    """
    day = pd.Timestamp(day)
    # Already cached and carrying data — nothing to do.
    if day in FWI.index and FWI.loc[day].notna().any():
        return True

    vals = {}
    for s in STIDS:
        v = _station_day_fwi(s, day)
        if v is not None:
            vals[s] = v
    if not vals:
        return False

    FWI.loc[day] = pd.Series(vals).reindex(STIDS)
    FWI.sort_index(inplace=True)
    FWI.to_parquet(FWI_PQ)
    print(f"live fetch {day.date()}: {len(vals)}/{len(STIDS)} stations")
    return True

In [ ]:
# =============================================================================
# Cell 4 — Derive the slope and FWI normalisation caps
# =============================================================================
# Prints both full distributions, then sets each cap at its configured percentile
# of the observed values so the top tail saturates rather than being compressed.
# SLOPE_CAP_PCT and FWI_CAP_PCT are set in Cell 1 and validated against PCTS.
#
# The slope sample is exported as a batch task rather than pulled with getInfo,
# because interactive compute times out well before 10 m sampling across the
# borough finishes. The sample and the resulting percentiles are both keyed, so
# changing the DEM, the region, the count or the seed rebuilds them.
# =============================================================================
RECALIBRATE = False     # True forces a re-export and re-derivation for one run

# 3DEP mosaic forced onto the working projection and pixel size, so slope is
# derived on the same grid and resolution the cap is later applied on.
dem   = (ee.ImageCollection(DEM_ID).select("elevation").mosaic()
         .setDefaultProjection(WORK_CRS, None, WORK_SCALE))
slope = ee.Terrain.slope(dem).rename("slope")

if RECALIBRATE or not SLOPE_JSON.exists():
    _fc = slope.sample(
        region=ee.Image(VEG_ASSET).geometry().bounds(),
        scale=WORK_SCALE, projection=WORK_CRS,
        numPixels=SLOPE_SAMPLES, seed=SLOPE_SEED,
        geometries=False, tileScale=8
    )

    stem = f"slope_sample_{sig(SLOPE_SAMPLES, SLOPE_SEED)}"
    src = run_export(_fc, stem, ["slope"])

    vals = pd.read_csv(src)["slope"].dropna().to_numpy(float)
    assert len(vals) > 1000, f"suspiciously small slope sample: {len(vals)}"
    print(f"slope sample n={len(vals):,}")
    slope_p = {q: float(np.percentile(vals, q)) for q in PCTS}
    slope_max = float(np.max(vals))
    SLOPE_JSON.write_text(json.dumps(
        {"slope_pct": slope_p, "n": len(vals), "max": slope_max}, indent=2))
else:
    _s = json.loads(SLOPE_JSON.read_text())
    # JSON keys come back as strings; the lookup below uses floats.
    slope_p = {float(k): v for k, v in _s["slope_pct"].items()}
    slope_max = _s["max"]

print("SLOPE (degrees) distribution:")
for q in PCTS:
    print(f"  p{q:<3} {slope_p[q]:.1f}")
print(f"  max  {slope_max:.1f}")

# FWI percentiles are pooled: the whole date x station table is flattened into
# one array first, so a percentile is "the value exceeded by X% of all station-
# days", not an average of per-station percentiles. Stations with a longer
# record therefore contribute more to the distribution, which is intended —
# the cap should reflect how often high FWI actually occurs across the network.
# nanpercentile skips the ffill gaps.
fwi_arr = FWI.to_numpy(dtype=float)
fwi_p = {q: float(np.nanpercentile(fwi_arr, q)) for q in PCTS}
print("\nFWI distribution (pooled across stations):")
for q in PCTS:
    print(f"  p{q:<3} {fwi_p[q]:.1f}")
print(f"  max  {float(np.nanmax(fwi_arr)):.1f}")

print("\nFWI max by station:")
print(FWI.max().round(1).to_string())

if RECALIBRATE or not CAPS_JSON.exists():
    SLOPE_CAP = float(slope_p[SLOPE_CAP_PCT])
    FWI_CAP   = float(fwi_p[FWI_CAP_PCT])
    assert SLOPE_CAP > 0 and FWI_CAP > 0, f"degenerate caps: slope {SLOPE_CAP}, FWI {FWI_CAP}"
    CAPS_JSON.write_text(json.dumps(
        {"SLOPE_CAP": SLOPE_CAP, "FWI_CAP": FWI_CAP,
        "SLOPE_CAP_PCT": SLOPE_CAP_PCT, "FWI_CAP_PCT": FWI_CAP_PCT,
        "slope_pct": slope_p, "fwi_pct": fwi_p}, indent=2))
else:
    _d = json.loads(CAPS_JSON.read_text())
    SLOPE_CAP, FWI_CAP = _d["SLOPE_CAP"], _d["FWI_CAP"]

print(f"\nSLOPE_CAP = {SLOPE_CAP:.1f} deg  (p{SLOPE_CAP_PCT})")
print(f"FWI_CAP   = {FWI_CAP:.1f}       (p{FWI_CAP_PCT})")

In [ ]:
# =============================================================================
# Cell 5 — Static component rasters (0-100 each)
# =============================================================================
# Fuel is a direct class lookup. Ember exposure is the mean fuel score in a
# 15-100 m annulus. Slope is rescaled against SLOPE_CAP.
#
# Nothing here is exported yet; this cell only defines the images and the cache
# name. Cell 6 does the export.
# =============================================================================
assert "slope" in globals() and "SLOPE_CAP" in globals(), "run Cell 4 first"

# remap turns each class code into its fuel score. Classes not in the table fall
# to defaultValue 0, so an unmapped code behaves like water rather than throwing.
veg = ee.Image(VEG_ASSET).select("b1").setDefaultProjection(WORK_CRS, None, WORK_SCALE)
fuel = veg.remap(list(VEGETATION_TO_WEIGHT.keys()),
                 list(VEGETATION_TO_WEIGHT.values()),
                 defaultValue=0).clamp(0, 100).rename("FUEL")

def circle_mean(img, radius):
    """Mean of img over a disc of the given radius, centred on each pixel."""
    return img.reduceNeighborhood(
        ee.Reducer.mean(), ee.Kernel.circle(radius=radius, units="meters"))


# Exposure is the mean of the annulus, recovered from two disc means. The inner
# disc mean is scaled by the ratio of the disc areas (k) and subtracted, then the
# remainder is divided by the remaining area fraction (1 - k). Without this the
# parcel's own fuel would be included and the layer would collapse toward FUEL.
#
#   annulus_mean = (disc_out_mean - k * disc_in_mean) / (1 - k)
#
# where k = area_in / area_out. The result is a true mean over the ring only.
k = (R_IN ** 2) / (R_OUT ** 2)
expo = (circle_mean(fuel, R_OUT)
        .subtract(circle_mean(fuel, R_IN).multiply(k))
        .divide(1 - k)
        .clamp(0, 100)
        .rename("EXPO"))

slope_pct = slope.multiply(100.0 / SLOPE_CAP).clamp(0, 100).rename("SLOPE_PCT")

# Static part of the risk equation: the three static components already carry
# their weights, so this is a 0-65 band that Cell 8 adds the daily FWI to.
static = (fuel.multiply(W_FUEL)
          .add(expo.multiply(W_EXPO))
          .add(slope_pct.multiply(W_SLOPE))
          .rename("STATIC"))

# Cache name hashes every input the contents depend on
ATTR_SIG = sig(json.dumps(VEGETATION_TO_WEIGHT, sort_keys=True),
               W_FUEL, W_EXPO, W_SLOPE, f"{SLOPE_CAP:.4f}",
               R_OUT, R_IN,
               VEG_ASSET, PARCEL_ASSET, WORK_CRS, WORK_SCALE)
ATTR_CSV = EXP / f"parcel_static_{ATTR_SIG}.csv"

print("static bands:", static.bandNames().getInfo())
print("static cache:", ATTR_CSV.name)

In [ ]:
# =============================================================================
# Cell 6 — Per-parcel means of the static components
# =============================================================================
# One mean per parcel per component, exported to CSV. Means are taken over each
# parcel's 10 m pixels on the equal-area working projection, so they are
# area-weighted by construction.
# =============================================================================
assert "static" in globals(), "run Cell 5 first"

parcels = ee.FeatureCollection(PARCEL_ASSET)
_N_PARCELS = parcels.size().getInfo()

# Convex hull plus a 150 m buffer. Used only to clip the map layers in Cell 9 —
# it is far cheaper to render than the parcel outlines and covers every parcel
# edge, so nothing is trimmed.
_BOUND = (parcels.geometry().convexHull(100)
          .transform(WORK_CRS, 1).buffer(150).transform("EPSG:4326", 1))

if not ATTR_CSV.exists():
    # STATIC plus its three components unweighted, so the CSV carries both the
    # combined band and the inputs for the click readout.
    stack = (static.addBands(fuel.rename("FUEL"))
                   .addBands(expo.rename("EXPO"))
                   .addBands(slope_pct.rename("SLOPE_PCT")))

    SEL = ["PAN", "STATIC", "FUEL", "EXPO", "SLOPE_PCT"]

    src = run_export(
        # Mean of every band over each parcel's pixels. PAN is a property, so it
        # passes through untouched. retainGeometry=False drops the outlines, which
        # keeps the CSV small and is all the scoring needs.
        stack.reduceRegions(
            collection=parcels, reducer=ee.Reducer.mean(),
            scale=WORK_SCALE, crs=WORK_CRS, tileScale=8
        ).select(SEL, retainGeometry=False),
        ATTR_CSV.stem, SEL)

    # The export lands under a Drive-generated name; move it to the keyed one so
    # the cache is found next run.
    if src != ATTR_CSV.resolve():
        shutil.copy(src, ATTR_CSV)

    _n = len(pd.read_csv(ATTR_CSV))
    assert _n > 0.9 * _N_PARCELS, f"static export short: {_n} of {_N_PARCELS} parcels"
else:
    print("reusing", ATTR_CSV)

In [ ]:
# =============================================================================
# Cell 7 — Per-parcel station area fractions
# =============================================================================
# Builds the nearest-station index directly rather than reading it from an asset,
# so the notebook has no dependency on an export from the FWI work.
#
# Every pixel is assigned to its nearest station. Distances are squared metres in
# the working CRS and the winner is tracked with a running minimum, which yields
# one index per pixel and resolves ties to the lower index. Each index then
# becomes a 0/1 band whose parcel mean is the share of the parcel nearest that
# station.
#
# This is a Voronoi partition built in image space. Doing it as images avoids a
# vector Voronoi (a separate geometry library, and polygons that would not line
# up with the 10 m grid) and avoids a 13-band reduceNeighborhood (far more
# expensive). One pass per station over two constant images is the whole cost,
# because each station only ever overwrites pixels it is closer to.
#
# The output is the weight matrix the FWI is later averaged with: for parcel p
# and station s, the value is the share of p's area for which s is nearest. The
# fractions sum to 1 for every covered parcel.
# =============================================================================

assert "parcels" in globals(), "run Cell 6 first"

# The grid distances are measured on. atScale pins the projection to the working
# pixel size so pixelCoordinates steps at exactly WORK_SCALE.
_proj = ee.Projection(WORK_CRS).atScale(WORK_SCALE)

# pixelCoordinates returns x/y in *pixel* units of _proj, not metres, so both are
# multiplied by the pixel size to get metres. Only differences of these images
# are ever taken, so the origin does not matter and the units cancel into m^2.
_xy = ee.Image.pixelCoordinates(_proj)
_X  = _xy.select("x").multiply(WORK_SCALE)
_Y  = _xy.select("y").multiply(WORK_SCALE)

# Each station's point, reprojected into the working CRS once. STATION_META is
# (lat, lon) but ee.Geometry.Point wants (lon, lat), hence the swap.
_STATION_XY = {
    s: ee.Geometry.Point([STATION_META[s][1], STATION_META[s][0]])
          .transform(WORK_CRS, 1).coordinates()
    for s in STIDS
}

# Running state for the single pass below: the smallest squared distance found so
# far at each pixel, and the index of the station that produced it. 1e30 is just
# "further than any real station", so the first station always wins.
best_d2 = ee.Image(1e30).toDouble()
best_i  = ee.Image(0)

for i, s in enumerate(STIDS, start=1):
    c  = _STATION_XY[s]
    # Constant images holding this station's x and y, so the distance is one
    # arithmetic expression over the whole grid.
    sx = ee.Image.constant(ee.Number(c.get(0)))
    sy = ee.Image.constant(ee.Number(c.get(1)))
    # Squared distance, not distance: the square root is monotonic, so it does
    # not change which station is nearest, and skipping it saves a sqrt.
    d2 = _X.subtract(sx).pow(2).add(_Y.subtract(sy).pow(2)).toDouble()

    # Strictly-less-than means an exact tie keeps the station with the lower
    # index, which makes the assignment deterministic. where() only overwrites
    # pixels this station is closer to, so the distance and the index stay in
    # sync pixel for pixel.
    closer  = d2.lt(best_d2)
    best_d2 = best_d2.where(closer, d2)
    best_i  = best_i.where(closer, i)

# sid is 1..13, the 1-based position in STIDS. toByte covers 13 values and keeps
# the exported bands small.
sid = best_i.toByte().rename("sid")

FRAC_SIG = sig("|".join(STIDS),
               json.dumps(STATION_META, sort_keys=True),
               PARCEL_ASSET, WORK_CRS, WORK_SCALE)
FRAC_CSV = EXP / f"parcel_station_frac_{FRAC_SIG}.csv"

SEL_F = ["PAN"] + FCOLS

# One 0/1 band per station: 1 where that station won the pixel, 0 elsewhere.
# Averaging each band over a parcel gives the fraction of the parcel for which
# that station is nearest, and the thirteen fractions sum to 1 per parcel.
frac_bands = ee.Image.cat([sid.eq(i).rename(c) for i, c in enumerate(FCOLS, start=1)])

if not FRAC_CSV.exists():
    src = run_export(
        frac_bands.reduceRegions(
            collection=parcels, reducer=ee.Reducer.mean(),
            scale=WORK_SCALE, crs=WORK_CRS, tileScale=8
        ).select(SEL_F, retainGeometry=False),
        FRAC_CSV.stem, SEL_F)

    if src != FRAC_CSV.resolve():
        shutil.copy(src, FRAC_CSV)

    _n = len(pd.read_csv(FRAC_CSV))
    assert _n > 0.9 * _N_PARCELS, f"frac export short: {_n} of {_N_PARCELS} parcels"
else:
    print("reusing", FRAC_CSV)

In [ ]:
# =============================================================================
# Cell 8 — Assemble the parcel table and score one day
# =============================================================================
# Parcel FWI is the area-weighted dot product of the station fractions with that
# day's values, rescaled against FWI_CAP. Missing stations are dropped and the
# remaining fractions renormalised, so a silent station does not drag its parcels
# toward zero.
#
# Both cache names are recomputed and compared first, so changing a weight, the
# vegetation table or the station set without rebuilding fails loudly instead of
# scoring against a table baked with the old settings.
#
# STN records the station holding the largest share of each parcel and STN_SHARE
# that share, so any FWI value can be attributed back to a station. STN_ABSENT
# flags the case where that station has no reading on the chosen date.
#
# NO_VEG flags parcels with essentially no vegetated area; they still receive a
# score from explosure, slope and FWI alone. PAN is coerced to int64 on both sides so the
# merge cannot silently go empty on a dtype split.
# =============================================================================
assert "SLOPE_CAP" in globals() and "FWI_CAP" in globals() and "FWI" in globals(), "run Cell 4 first"
assert ATTR_CSV.exists() and FRAC_CSV.exists(), "run Cells 6 and 7"

# Recompute the expected static cache name from the current settings and require
# it to match. This is the guard against scoring with a table baked under
# different weights, a different table or a different cap.
_expect = EXP / (f"parcel_static_"
                 f"{sig(json.dumps(VEGETATION_TO_WEIGHT, sort_keys=True), W_FUEL, W_EXPO, W_SLOPE, f'{SLOPE_CAP:.4f}', R_OUT, R_IN, VEG_ASSET, PARCEL_ASSET, WORK_CRS, WORK_SCALE)}"
                 f".csv")
assert ATTR_CSV == _expect, (
    f"weights or table changed — rerun Cells 5 and 6. expected {_expect.name}")

_fexpect = EXP / (f"parcel_station_frac_"
                  f"{sig('|'.join(STIDS), json.dumps(STATION_META, sort_keys=True), PARCEL_ASSET, WORK_CRS, WORK_SCALE)}"
                  f".csv")
assert FRAC_CSV == _fexpect, (
    f"station set or metadata changed — rerun Cell 7. expected {_fexpect.name}")

attr = pd.read_csv(ATTR_CSV)
frac = pd.read_csv(FRAC_CSV)

# Both sides key on PAN. Coercing to int64 makes the merge exact: a PAN read back
# as 182028.0 would not match an int 182028, and the merge would come back empty
# with no error raised.
for _d in (attr, frac):
    _d["PAN"] = pd.to_numeric(_d["PAN"], errors="coerce")
    _d.dropna(subset=["PAN"], inplace=True)
    _d["PAN"] = _d["PAN"].astype("int64")

# 999999 is the parcel table's placeholder row, not a real parcel.
attr = attr.query("PAN != 999999").drop_duplicates("PAN")
frac = frac.query("PAN != 999999").drop_duplicates("PAN")

# Inner join: a parcel needs both its static components and its station weights
# to be scorable. Anything present in only one export drops out here.
haz  = attr.merge(frac, on="PAN", how="inner")
assert len(haz) > 0, "no parcels survived the merge — check PAN dtype"

# FUEL under 1 point (effectively all water or bare) is flagged, not dropped.
# These parcels still get a score from slope and FWI.
haz["NO_VEG"] = haz["FUEL"].fillna(0).lt(1).astype(int)

# Dominant station per parcel, from the largest area fraction. "-" where a parcel
# has no station coverage at all, since argmax on an all-zero row would otherwise
# silently return station 1.
_fr  = haz[FCOLS].fillna(0).to_numpy(dtype=float)
_dom = _fr.argmax(axis=1)
haz["STN"] = np.where(_fr.sum(axis=1) > 0, [STIDS[i] for i in _dom], "-")
haz["STN_SHARE"] = _fr.max(axis=1)


def score(day):
    """Score every parcel for one date, or None if nothing can be scored."""
    # Normalise to midnight — the index is date-only — then snap to the nearest
    # available date if this one has no record, rather than raising.
    day = pd.Timestamp(day).normalize()
    if day not in FWI.index:
        _v = FWI.dropna(how="all").index
        if len(_v) == 0:
            return None
        day = _v[_v.get_indexer([day], method="nearest")[0]]
    row = FWI.loc[day]

    # Stations with an actual reading. Everything else is excluded from both the
    # weights and the values, which is what stops a dead station pulling its
    # parcels down toward zero.
    have = [s for s in STIDS if pd.notna(row.get(s))]
    if not have:
        return None

    # The fraction columns matching those stations, in the same order.
    cols = [FCOLS[STIDS.index(s)] for s in have]

    # Weight matrix, one column per reporting station and one row per parcel:
    # the share of the parcel that station is nearest.
    w = haz[cols].fillna(0).to_numpy(dtype=float)

    # Total weight each parcel still has after the absent stations were dropped.
    # A parcel at 0 has no coverage from any reporting station and is skipped
    # rather than scored as 0.
    tot = w.sum(axis=1)
    ok = tot > 0

    # Station FWI rescaled to 0-100 against the cap and clipped, so the cap and
    # above saturates. Order matches `have`.
    vals = np.clip(np.array([row[s] for s in have], dtype=float) * (100.0 / FWI_CAP), 0, 100)

    # Weighted mean, as a dot product over each parcel's row of weights divided
    # by that row's total. Dividing by tot is the renormalisation: it rescales
    # the surviving fractions back up to 1, so a parcel that lost half its
    # stations still gets a true mean of the remaining ones rather than a value
    # dragged toward 0. Parcels where ok is False stay NaN and drop out below.
    fwi_pct = np.full(len(haz), np.nan)
    fwi_pct[ok] = (w[ok] @ vals) / tot[ok]

    out = haz[["PAN", "FUEL", "EXPO", "SLOPE_PCT", "STATIC", "NO_VEG",
               "STN", "STN_SHARE"]].copy()
    out["FWI_PCT"] = fwi_pct
    out = out[out["FWI_PCT"].notna()].copy()
    if out.empty:
        return None

    # Flag parcels whose dominant station was one of the absent ones. They are
    # still scored, but from neighbours, so their value should not be read as
    # coming from the station named in STN.
    out["STN_ABSENT"] = ~out["STN"].isin(have)

    # STATIC already carries 0.65 of the weight; add the FWI's 0.35 on top and
    # clamp, since rounding or a clipped FWI cannot push past 100.
    out["RISK"] = (out["STATIC"] + out["FWI_PCT"] * W_FWI).clip(0, 100).round(1)
    out = out[out["RISK"].notna()].copy()
    if out.empty:
        return None
    return out

print(len(haz), "parcels with static attributes and station coverage")
print(f"caps in use: slope {SLOPE_CAP:.1f} deg (p{SLOPE_CAP_PCT}), "
      f"FWI {FWI_CAP:.1f} (p{FWI_CAP_PCT})")
print("spearman FUEL vs EXPO:",
      round(haz[["FUEL", "EXPO"]].corr("spearman").iloc[0, 1], 3))
print("parcels flagged NO_VEG:", int(haz["NO_VEG"].sum()))
print("dominant-station share, median:", round(float(haz["STN_SHARE"].median()), 3))

# QA against the worst case: the hottest date by network mean.
_s0 = score(FWI.mean(axis=1).idxmax())
if _s0 is not None:
    print("\nQA on hottest date:")
    print("  expo share at floor:", round(float((_s0["EXPO"] <= 0.01).mean()), 3))
    print("  parcels on an absent station:", int(_s0["STN_ABSENT"].sum()))
    print(_s0["RISK"].describe(percentiles=[.5, .9, .99]).round(1).to_string())

# -----------------------------------------------------------------------------
# Coverage diagnostic: find the date on which the most parcels have no reporting
# station at all. Those parcels cannot be scored that day and drop off the map,
# so this is the worst case for how much of the borough disappears at once.
# -----------------------------------------------------------------------------
def dark(day):
    """Return (row positions of uncovered parcels, stations reporting) for a date."""
    row = FWI.loc[pd.Timestamp(day)]
    have = [s for s in STIDS if pd.notna(row.get(s))]
    if not have:
        return None
    cols = [FCOLS[STIDS.index(s)] for s in have]
    # Surviving coverage per parcel; zero means nothing reporting covers it.
    tot = haz[cols].fillna(0).to_numpy(float).sum(axis=1)
    return np.flatnonzero(tot == 0), have

# Scan every date and keep the one with the largest uncovered set.
best = (0, None, None)
for d in FWI.index:
    r = dark(d)
    if r is None:
        continue
    idx, have = r
    if len(idx) > best[0]:
        best = (len(idx), d, idx)

n, d, idx = best
print(f"{n} parcels with no coverage on {d.date()}")
# A parcel lying wholly inside one station's area vanishes the moment that
# station goes silent, so the count tracks the single-station count closely.
print("single-station parcels in that set:",
      int((haz.iloc[idx]["STN_SHARE"] > 0.999).sum()))
print(haz.iloc[idx[:25]][["PAN", "STN", "STN_SHARE", "FUEL", "NO_VEG"]]
        .to_string())

In [ ]:
# =============================================================================
# Cell 9 — Risk map with date picker, search and parcel/station inspector
# =============================================================================
# Risk is painted onto the parcels as a single raster of class indices, so a date
# change costs one evaluation instead of five filtered polygon layers. Band
# thresholds are fixed so colour means the same thing on every date. Clicking a
# parcel reports every component, its point contribution, the station dominating
# its FWI, and the protection status of the ground it sits on.
#
# Parcels are re-added after the raster on each render, because addLayer appends
# to the top of the stack and the outlines would otherwise be buried.
#
# The picker runs to today rather than to the last cached date; render calls
# ensure_fwi() first, which pulls the day from AKFF and caches it. Parcel keys
# are rendered as zero-decimal strings on both sides of the dictionary so the
# lookup cannot miss on a float format difference.
# =============================================================================
from ipywidgets import DatePicker, HTML, VBox, HBox, Text, Button
from IPython.display import display
from google.colab import output
output.enable_custom_widget_manager()

BANDS = [(0, 25, "2c7bb6", "Low"),
         (25, 40, "abd9e9", "Moderate"),
         (40, 55, "ffffbf", "Elevated"),
         (55, 70, "fdae61", "High"),
         (70, 101, "d7191c", "Very high")]
PALETTE = [c for _, _, c, _ in BANDS]

# Protection codes carried by the management-option polygons, worst first.
POLICY = {
    "C": ("CRITICAL", "#FF0000"),
    "F": ("FULL",     "#FFA500"),
    "M": ("MODIFIED", "#CCCC00"),
    "L": ("LIMITED",  "#00AA00"),
    "U": ("UNPLANNED", "#888888"),
}

def valid_dates():
    """Dates that carry at least one station reading. Recomputed per render."""
    return FWI.dropna(how="all").index

def nearest_available(d):
    """Snap d to the closest date that has data."""
    d = pd.Timestamp(d)
    v = valid_dates()
    if d in v:
        return d
    return v[v.get_indexer([d], method="nearest")[0]]

def band_image(risk_img):
    # chained where: the highest satisfied threshold wins
    img = (ee.Image(0)
             .where(risk_img.gte(25), 1)
             .where(risk_img.gte(40), 2)
             .where(risk_img.gte(55), 3)
             .where(risk_img.gte(70), 4))
    # Unscored parcels carry -1; masking on gte(0) hides them instead of
    # painting them as Low.
    return img.updateMask(risk_img.gte(0)).rename("risk")

PARCELS_FC = ee.FeatureCollection(PARCEL_ASSET)
LOOKUP = {}
empty = ee.Image().byte()

# Management polygons are only used for the popup and the optional layer.
mgmt_zones = ee.FeatureCollection(MGMT_ASSET).filterBounds(_BOUND) if MGMT_ASSET else None

def parcel_outlines():
    return PARCELS_FC.style(color="000000", fillColor="00000000", width=0.6)

def station_fc(day):
    """Station points styled by whether they reported on `day`."""
    row = FWI.loc[pd.Timestamp(day)]
    feats = []
    for s in STIDS:
        v = row.get(s)
        up = pd.notna(v)
        col = "#0055ff" if up else "#888888"
        feats.append(ee.Feature(
            ee.Geometry.Point([STATION_META[s][1], STATION_META[s][0]]),
            {"stid": s, "fwi": float(v) if up else 0.0, "active": int(up),
             "style": {"color": col, "pointSize": 7, "width": 2, "fillColor": col}}))
    return ee.FeatureCollection(feats)

M = geemap.Map()
M.centerObject(PARCELS_FC, 10)
M.add_basemap("SATELLITE")
M.addLayer(dem.clip(_BOUND), {"min": 100, "max": 800,
                              "palette": ["black", "white"]}, "elevation", False)
M.addLayer(veg.clip(_BOUND), {"min": 0, "max": 17, "palette": VEG_PALETTE},
           "vegetation", False)
if mgmt_zones is not None:
    M.addLayer(empty.paint(mgmt_zones, 1, 2), {"palette": ["#222222"]},
               "mgmt boundaries", False)
M.addLayer(parcel_outlines(), {}, "parcels", opacity=0.35)
M.add_legend(title="Fire risk", keys=[n for _, _, _, n in BANDS], colors=PALETTE)

status = HTML("")
source_hint = HTML("<div style='font-size:11px;color:#777;padding:2px 6px'>"
                   "<i>FWI source: not yet loaded</i></div>")
box = HTML("<div style='padding:10px'>click a parcel, or search a PAN or address</div>")

search = Text(placeholder="PAN or address", description="Find:", layout={"width": "300px"})
go = Button(description="Search", button_style="primary")

def render(d):
    global LOOKUP
    d = pd.Timestamp(d)
    # Pull the date from AKFF if it is not cached yet.
    if not ensure_fwi(d):
        status.value = f"<div style='padding:6px'>{d.date()}: no station data</div>"
        return
    d = nearest_available(d)

    s = score(d)
    if s is None:
        status.value = f"<div style='padding:6px'>{d.date()}: no station data</div>"
        return
    # Rebuilt every render so the click readout always matches what is on screen.
    LOOKUP = s.set_index("PAN").to_dict("index")

    # Clear the previous raster, outlines and station points before drawing.
    for name in ("risk", "parcels", "stations", "fire perimeters"):
        for l in list(M.layers):
            if getattr(l, "name", "") == name:
                M.remove_layer(l)

    # Server-side dictionary of PAN -> risk. Keys are zero-decimal strings and
    # the join side formats PAN the same way, so no float/int mismatch can make
    # a parcel miss. Unlisted parcels fall to -1 and are masked by band_image.
    lk = ee.Dictionary({str(int(k)): float(v)
                        for k, v in s.set_index("PAN")["RISK"].items()})
    fc = PARCELS_FC.map(lambda f: f.set(
        "RISK", lk.get(ee.Number(f.get("PAN")).format("%d"), -1)))
    risk_img = fc.reduceToImage(["RISK"], ee.Reducer.first())

    M.addLayer(band_image(risk_img).clip(_BOUND),
               {"min": 0, "max": 4, "palette": PALETTE}, "risk", opacity=0.75)
    M.addLayer(parcel_outlines(), {}, "parcels", opacity=0.35)
    M.addLayer(station_fc(d).style(styleProperty="style"), {}, "stations")

    # Provenance: distinguishes a date served from the cached parquet from one
    # fetched live just now, and from the newest date in the record where the
    # distinction is ambiguous.
    cached = "cached" if d in FWI.index and d < FWI.index.max() else "live/edge"
    source_hint.value = (f"<div style='font-size:11px;color:#666;padding:2px 6px'>"
                         f"<i>FWI source ({d.date()}): {len([x for x in STIDS if pd.notna(FWI.loc[d].get(x))])}"
                         f"/{len(STIDS)} stations reporting, {cached}</i></div>")

    status.value = (f"<div style='padding:6px'><b>{d.date()}</b> — "
                    f"median {s['RISK'].median():.0f}, "
                    f"90th pct {s['RISK'].quantile(.9):.0f}, "
                    f"{len(s):,} parcels, {int(s['NO_VEG'].sum())} no-veg, "
                    f"{int(s['STN_ABSENT'].sum())} on absent station</div>")

def policy_at(pt):
    """Protection code and label for a point, defaulting to unplanned."""
    if mgmt_zones is None:
        return "U", POLICY["U"]
    try:
        hit = mgmt_zones.filterBounds(pt).first().getInfo()
        code = str(hit["properties"].get("PROT", "U")).strip() if hit else "U"
    except Exception:
        code = "U"
    return code, POLICY.get(code, POLICY["U"])

def show_parcel(pan, r, addr="", target=None):
    """Fill the inspector box for one parcel."""
    code, (label, colour) = policy_at(ee.Geometry.Point(
        [STATION_META.get(r["STN"], (64.84, -147.72))[1],
         STATION_META.get(r["STN"], (64.84, -147.72))[0]]))
    band = next((n for lo, hi, _, n in BANDS if lo <= r["RISK"] < hi), "-")
    flag = " <b>[NO VEG]</b>" if r["NO_VEG"] else ""
    absent = " — absent that day, neighbours used" if r["STN_ABSENT"] else ""
    where = f"<div style='color:#555;font-size:12px'>{addr}</div>" if addr else ""
    # FWI_PCT is on the 0-100 rescaled scale, so it is multiplied back by
    # FWI_CAP/100 to report the raw index value alongside its points.
    (target or box).value = (f"<div style='padding:10px;border-left:4px solid #d7191c;font-size:12px'>"
                 f"<b>PAN {pan}</b>{flag}{where}<br>"
                 f"<b>RISK {r['RISK']:.1f}</b> ({band})<hr>"
                 f"fuel {r['FUEL']:.0f} pts {r['FUEL']*W_FUEL:.1f}<br>"
                 f"exposure {r['EXPO']:.0f} pts {r['EXPO']*W_EXPO:.1f}<br>"
                 f"slope {r['SLOPE_PCT']:.0f} pts {r['SLOPE_PCT']*W_SLOPE:.1f}<br>"
                 f"FWI {r['FWI_PCT']*FWI_CAP/100:.1f} pts {r['FWI_PCT']*W_FWI:.1f}<hr>"
                 f"<span style='color:{colour};font-weight:bold'>"
                 f"protection: {label}</span><br>"
                 f"<span style='color:#555'>FWI dominated by {r['STN']} "
                 f"({r['STN_SHARE']*100:.0f}% of parcel){absent}</span></div>")

def on_click(**kw):
    if kw.get("type") != "click":
        return
    lat, lon = kw["coordinates"]
    pt = ee.Geometry.Point([lon, lat])

    # A station within 500 m wins over a parcel, so the points stay inspectable
    # even where they sit on a parcel.
    try:
        st = station_fc(nearest_available(pd.Timestamp.today())).filterBounds(pt.buffer(500))
        if st.size().getInfo() > 0:
            p = st.first().getInfo()["properties"]
            up = p.get("active") == 1
            box.value = (f"<div style='padding:10px;font-size:12px'>"
                         f"<b>station {p['stid']}</b><br>"
                         f"status: {'reporting' if up else 'no reading that day'}<br>"
                         f"FWI: {p['fwi']:.1f}" if up else
                         f"<div style='padding:10px;font-size:12px'>"
                         f"<b>station {p['stid']}</b><br>status: no reading that day</div>")
            return
    except Exception:
        pass

    # filterBounds is bbox-only, so confirm the point is actually inside the
    # polygon before accepting its PAN.
    pan = None
    try:
        cand = PARCELS_FC.filterBounds(pt).limit(10).toList(10).getInfo()
        for f in cand:
            if ee.Geometry(f["geometry"]).contains(pt).getInfo():
                pan = int(f["properties"]["PAN"])
                break
    except Exception:
        pan = None

    if pan is None or pan not in LOOKUP:
        box.value = "<div style='padding:10px'>no scored parcel here</div>"
        return
    show_parcel(pan, LOOKUP[pan])

def do_search(b):
    q = search.value.strip()
    if not q:
        return
    box.value = f"<div style='padding:10px'>searching {q} …</div>"

    # PAN first, trying the padded and unpadded forms the table uses.
    if q.replace("-", "").isdigit():
        pan = int(q.replace("-", ""))
        if pan in LOOKUP:
            feat = PARCELS_FC.filter(ee.Filter.eq("PAN", pan)).first()
            M.centerObject(feat, 17)
            show_parcel(pan, LOOKUP[pan])
            return
        box.value = f"<div style='padding:10px'>PAN {pan} is not scored on this date</div>"
        return

    # Otherwise geocode, then take the parcel under the point.
    try:
        loc = geocoder.geocode(q if "ak" in q.lower() else f"{q}, Fairbanks, AK",
                               timeout=10)
    except Exception:
        loc = None
    if loc is None:
        box.value = f"<div style='padding:10px'>not found: {q}</div>"
        return

    pt = ee.Geometry.Point([loc.longitude, loc.latitude])
    M.centerObject(pt, 16)
    try:
        cand = PARCELS_FC.filterBounds(pt).limit(5).toList(5).getInfo()
    except Exception:
        cand = []
    for f in cand:
        if ee.Geometry(f["geometry"]).contains(pt).getInfo():
            pan = int(f["properties"]["PAN"])
            if pan in LOOKUP:
                show_parcel(pan, LOOKUP[pan], loc.address.split(",")[0])
            else:
                box.value = f"<div style='padding:10px'>parcel {pan} is not scored on this date</div>"
            return
    box.value = f"<div style='padding:10px'>{loc.address.split(',')[0]} — no parcel here</div>"

go.on_click(do_search)
M.on_interaction(on_click)

V = valid_dates()
LAST = V.max()
picker = DatePicker(value=LAST.date(), min=V.min().date(),
                    max=max(LAST, pd.Timestamp.today().normalize()).date(),
                    description="date")
picker.observe(lambda c: c["type"] == "change" and c["new"] and render(c["new"]), names="value")

render(LAST)
display(VBox([source_hint, HBox([picker, search, go]), M, status, box]))

In [ ]:
# =============================================================================
# Cell 10 — Historical fire perimeters and a validation readout
# =============================================================================
# Pulls Alaska Fire Service fire history for the borough envelope, falls back to
# MTBS if the AFS service returns nothing usable, and compares the score on a real
# fire's perimeter ring against a cool day and against the borough mean.
#
# The AFS query asks for everything in the envelope and filters locally, because
# the service rejects a where clause built from guessed field names. Field names
# are printed so a future tightening can use real ones.
# =============================================================================
AFS_QUERY = ("https://fire.ak.blm.gov/arcgis/rest/services/"
             "MapAndFeatureServices/FireHistory/MapServer/1/query")

fires = None
fire_source = None

# Probe the schema once. Harmless if it fails, and it tells you the real names.
try:
    meta = requests.get(AFS_QUERY.rsplit("/query", 1)[0],
                        params={"f": "json"}, timeout=20).json()
    print("AFS fields:", [f["name"] for f in meta.get("fields", [])])
except Exception as exc:
    print("AFS field probe failed:", exc)

try:
    r = requests.get(AFS_QUERY, timeout=30, params={
        "where": "1=1",
        "geometry": "-150.2,64.2,-145.0,65.6",
        "geometryType": "esriGeometryEnvelope", "inSR": "4326",
        "spatialRel": "esriSpatialRelIntersects",
        "outFields": "*", "outSR": "4326",
        "returnGeometry": "true", "resultRecordCount": 1000, "f": "geojson"})
    print("AFS status", r.status_code)
    if r.status_code == 200 and r.text.strip().startswith("{"):
        gj = r.json()
        keep = []
        for f in gj.get("features", []):
            p = {str(k).upper(): v for k, v in f.get("properties", {}).items()}
            try:
                yr, ac = int(p.get("FIREYEAR")), float(p.get("ACRES") or 0)
            except (TypeError, ValueError):
                continue
            if yr >= 2000 and ac > 20:
                keep.append(f)
        print(f"AFS usable features: {len(keep)}")
        if keep:
            gj["features"] = keep
            fires = geemap.geojson_to_ee(gj).filterBounds(_BOUND)
            fire_source = "BLM Alaska Fire History (2000+, >20 ac)"
        else:
            print("AFS error body:", str(gj)[:300])
    else:
        print("AFS body:", r.text[:300])
except Exception as exc:
    print("AFS query failed:", exc)

# MTBS fallback, which carries exact ignition dates.
if fires is None:
    fires = (ee.FeatureCollection("USFS/GTAC/MTBS/burned_area_boundaries/v1")
               .filterBounds(_BOUND)
               .filter(ee.Filter.gte("Ig_Date", 946684800000)))
    fire_source = "MTBS (2000+)"

# Keep the raw feature dicts so the target is picked by position. Names repeat
# across years and the field differs by source, so a name filter is unreliable.
feats = fires.limit(40).getInfo().get("features", [])
assert feats, "no perimeters returned"

rows = []
for i, f in enumerate(feats):
    p = f.get("properties", {})
    name = p.get("NAME") or p.get("Incid_Name") or p.get("FIRENAME") or "Unnamed"
    acres = float(p.get("ACRES") or p.get("BurnBndAc") or 0)
    if p.get("FIREYEAR"):
        when = str(int(p["FIREYEAR"]))
    elif isinstance(p.get("Ig_Date"), (int, float)):
        when = pd.to_datetime(p["Ig_Date"], unit="ms").strftime("%Y-%m-%d")
    else:
        when = "unknown"
    rows.append({"i": i, "fire": name, "when": when, "acres": round(acres, 1)})

fire_df = pd.DataFrame(rows).sort_values(["when", "acres"], ascending=False)
print(f"{len(rows)} perimeters from {fire_source}")
display(fire_df.head(15))

if "fire perimeters" in [l.name for l in M.layers if hasattr(l, "name")]:
    M.remove_layer("fire perimeters")
M.addLayer(empty.paint(fires, 1, 3), {"palette": ["#ff5500"]}, "fire perimeters")

# Default target for Cell 11's picker: the largest fire in the window.
top = fire_df.iloc[0]
if re.fullmatch(r"\d{4}-\d{2}-\d{2}", top["when"]):
    hot_day = pd.Timestamp(top["when"])
else:
    hot_day = pd.Timestamp(f"{top['when']}-07-15")
print(f"Cell 11 will open on {hot_day.date()} ({top['fire']})")

In [ ]:
# =============================================================================
# Cell 11 — Risk surface under fire perimeters, with a steppable date
# =============================================================================
# Same controls as Cell 9 on its own map: date picker, search, parcel and station
# inspector, optional layers. Two additions. Perimeters are drawn on top and
# coloured by how far their ignition is from the selected date, so stepping the
# picker backwards walks the run-up to each fire. And clicking a perimeter reports
# its name, ignition date and acreage.
#
# Offset bands are ±1 day, ±7 days and ±30 days. A perimeter that fired more than
# a month either side of the selected date is not drawn, and is not clickable
# either, so a hit can never land on a fire you cannot see. Fires dated only to a
# year (the AFS record) are placed at 15 July and marked approximate.
#
# Hit-testing is local: shapely polygons for the perimeters and the station
# coordinate table for the points, so a click costs no server round trips until
# it falls through to a parcel.
# =============================================================================
import math
from shapely.geometry import shape, Point

# Date-keyed perimeter collection. Built from the features Cell 10 already
# fetched, so nothing is re-queried.
_fire_rows = []
for _f in feats:
    _p = _f.get("properties", {})
    if isinstance(_p.get("Ig_Date"), (int, float)) and _p["Ig_Date"]:
        _ts, _approx = pd.to_datetime(_p["Ig_Date"], unit="ms"), False
    elif _p.get("FIREYEAR"):
        _ts, _approx = pd.Timestamp(f"{int(_p['FIREYEAR'])}-07-15"), True
    else:
        continue
    _fire_rows.append((_f, _ts, _approx))

assert _fire_rows, "no perimeters carry a usable date"

fire_geo = ee.FeatureCollection([

    ee.Feature(f["geometry"], {
        "fname": f["properties"].get("NAME")
                 or f["properties"].get("Incid_Name") or "Unnamed",
        "acres": float(f["properties"].get("ACRES")
                       or f["properties"].get("BurnBndAc") or 0),
        # wms is epoch milliseconds so the date bands can be filtered
        # server-side with a numeric range instead of parsing date strings.
        "wms":   int(ts.value // 10**6),
        "wstr":  ts.strftime("%Y-%m-%d") + (" (approx)" if approx else ""),
    })
    for f, ts, approx in _fire_rows
])

# Local shapely copies, so a click never has to ask the server where a fire is.
# buffer(0) repairs invalid rings before they reach the contains test.
_fire_shapes = []
for _f, _ts, _approx in _fire_rows:
    try:
        _g = shape(_f["geometry"]).buffer(0)
    except Exception:
        continue
    _fire_shapes.append({
        "name":  _f["properties"].get("NAME")
                 or _f["properties"].get("Incid_Name") or "Unnamed",
        "acres": float(_f["properties"].get("ACRES")
                       or _f["properties"].get("BurnBndAc") or 0),
        "wstr":  _ts.strftime("%Y-%m-%d") + (" (approx)" if _approx else ""),
        "wms":   int(_ts.value // 10**6),
        "geom":  _g,
    })

LOOKUP2 = {}
CUR_DAY = hot_day          # date currently drawn; set by render2
box2 = HTML("<div style='padding:10px'>click a parcel, a station, or a perimeter</div>")
status2 = HTML("")
hint2 = HTML("<div style='font-size:11px;color:#777;padding:2px 6px'>"
             "<i>perimeters shown: none</i></div>")
search2 = Text(placeholder="PAN or address", description="Find:", layout={"width": "300px"})
go2 = Button(description="Search", button_style="primary")

F = geemap.Map()
F.centerObject(PARCELS_FC, 10)
F.add_basemap("SATELLITE")
F.addLayer(dem.clip(_BOUND), {"min": 100, "max": 800,
                              "palette": ["black", "white"]}, "elevation", False)
F.addLayer(veg.clip(_BOUND), {"min": 0, "max": 17, "palette": VEG_PALETTE},
           "vegetation", False)
if mgmt_zones is not None:
    F.addLayer(empty.paint(mgmt_zones, 1, 2), {"palette": ["#222222"]},
               "mgmt boundaries", False)
F.add_legend(title="Fire risk", keys=[n for _, _, _, n in BANDS], colors=PALETTE)

def render2(d):
    global LOOKUP2, CUR_DAY
    d = pd.Timestamp(d)
    if not ensure_fwi(d):
        status2.value = f"<div style='padding:6px'>{d.date()}: no station data</div>"
        return
    d = nearest_available(d)
    CUR_DAY = d

    s = score(d)
    if s is None:
        status2.value = f"<div style='padding:6px'>{d.date()}: no station data</div>"
        return
    LOOKUP2 = s.set_index("PAN").to_dict("index")

    for name in ("risk", "parcels", "stations", "fire 1d", "fire 7d", "fire 30d"):
        for l in list(F.layers):
            if getattr(l, "name", "") == name:
                F.remove_layer(l)

    lk = ee.Dictionary({str(int(k)): float(v)
                        for k, v in s.set_index("PAN")["RISK"].items()})
    fc = PARCELS_FC.map(lambda f: f.set(
        "RISK", lk.get(ee.Number(f.get("PAN")).format("%d"), -1)))
    risk_img = fc.reduceToImage(["RISK"], ee.Reducer.first())

    F.addLayer(band_image(risk_img).clip(_BOUND),
               {"min": 0, "max": 4, "palette": PALETTE}, "risk", opacity=0.75)
    F.addLayer(parcel_outlines(), {}, "parcels", opacity=0.3)
    F.addLayer(station_fc(d).style(styleProperty="style"), {}, "stations")

    # Perimeters by distance from the selected date. The closest band is added
    # last so the tightest offset sits on top of the wider ones.
    ms = int(d.value // 10**6)
    shown = 0
    for label, days, colour, width in (("30d", 30, "#ffe082", 1),
                                       ("7d", 7, "#ff9800", 2),
                                       ("1d", 1, "#e53935", 3)):
        band = fire_geo.filter(ee.Filter.rangeContains(
            "wms", ms - days * 86400000, ms + days * 86400000))
        F.addLayer(empty.paint(band, 1, width), {"palette": [colour]},
                   f"fire {label}")
        shown += band.size().getInfo()

    hint2.value = (f"<div style='font-size:11px;color:#666;padding:2px 6px'>"
                   f"<span style='color:#e53935'>■</span> ±1d "
                   f"<span style='color:#ff9800'>■</span> ±7d "
                   f"<span style='color:#ffe082'>■</span> ±30d — "
                   f"{shown} perimeters near {d.date()}</div>")

    status2.value = (f"<div style='padding:6px'><b>{d.date()}</b> — "
                     f"median {s['RISK'].median():.0f}, "
                     f"90th pct {s['RISK'].quantile(.9):.0f}, "
                     f"{len(s):,} parcels, "
                     f"{int(s['STN_ABSENT'].sum())} on absent station</div>")

def fire_at(lon, lat, day):
    """Perimeter containing the point, else the nearest within 1.5 km.

    Only perimeters inside the ±30 day band are considered, so a click cannot
    land on a fire that is not currently drawn.
    """
    pt = Point(lon, lat)
    ms = int(pd.Timestamp(day).value // 10**6)
    span = 30 * 86400000
    near = None
    for fr in _fire_shapes:
        if abs(fr["wms"] - ms) > span:
            continue
        if fr["geom"].contains(pt):
            return {**fr, "dist": 0.0}
        d = fr["geom"].distance(pt)      # degrees
        if near is None or d < near[0]:
            near = (d, fr)
    if near and near[0] * 111_000 < 1500:
        return {**near[1], "dist": near[0] * 111_000}
    return None

def station_at(lon, lat, day):
    """Nearest station within 800 m, from the local coordinate table."""
    # Normalise to midnight — the index is date-only — then snap to a date that
    # actually exists, since the picker can sit on a day with no record.
    day = pd.Timestamp(day).normalize()
    if day not in FWI.index:
        day = nearest_available(day)
    row = FWI.loc[day]
    best = None
    for s, (la, lo) in STATION_META.items():
        dm = (((la - lat) * 111_000) ** 2
              + ((lo - lon) * 111_000 * math.cos(math.radians(lat))) ** 2) ** 0.5
        if best is None or dm < best[0]:
            best = (dm, s)
    if best and best[0] < 800:
        s = best[1]
        v = row.get(s)
        return {"stid": s, "active": pd.notna(v),
                "fwi": float(v) if pd.notna(v) else 0.0}
    return None

def on_click2(**kw):
    if kw.get("type") != "click":
        return
    lat, lon = kw["coordinates"]
    pt = ee.Geometry.Point([lon, lat])

    fp = fire_at(lon, lat, CUR_DAY)
    if fp:
        box2.value = (f"<div style='padding:10px;font-size:12px;"
                      f"border-left:4px solid #e53935'>"
                      f"<b>{fp['name']}</b><br>"
                      f"ignition: {fp['wstr']}<br>"
                      f"{fp['acres']:,.0f} acres<br>"
                      f"<span style='color:#555'>{fp['dist']/1000:.1f} km from click</span></div>")
        return

    st = station_at(lon, lat, CUR_DAY)
    if st:
        box2.value = (f"<div style='padding:10px;font-size:12px'>"
                      f"<b>station {st['stid']}</b><br>"
                      f"status: {'reporting' if st['active'] else 'no reading'}<br>"
                      f"FWI: {st['fwi']:.1f}</div>")
        return

    # One call: the geometry test runs server-side instead of once per candidate.
    try:
        hit = PARCELS_FC.filter(
            ee.Filter.contains(leftField=".geo", rightValue=pt)).first().getInfo()
    except Exception:
        hit = None

    if not hit:
        box2.value = "<div style='padding:10px'>no scored parcel here</div>"
        return
    pan = int(hit["properties"]["PAN"])
    if pan not in LOOKUP2:
        box2.value = "<div style='padding:10px'>parcel is not scored on this date</div>"
        return
    show_parcel(pan, LOOKUP2[pan], target=box2)

def do_search2(b):
    q = search2.value.strip()
    if not q:
        return
    box2.value = f"<div style='padding:10px'>searching {q} …</div>"

    if q.replace("-", "").isdigit():
        pan = int(q.replace("-", ""))
        if pan in LOOKUP2:
            F.centerObject(PARCELS_FC.filter(ee.Filter.eq("PAN", pan)).first(), 17)
            show_parcel(pan, LOOKUP2[pan], target=box2)
        else:
            box2.value = f"<div style='padding:10px'>PAN {pan} is not scored on this date</div>"
        return

    try:
        loc = geocoder.geocode(q if "ak" in q.lower() else f"{q}, Fairbanks, AK",
                               timeout=10)
    except Exception:
        loc = None
    if loc is None:
        box2.value = f"<div style='padding:10px'>not found: {q}</div>"
        return

    pt = ee.Geometry.Point([loc.longitude, loc.latitude])
    F.centerObject(pt, 16)
    try:
        hit = PARCELS_FC.filter(
            ee.Filter.contains(leftField=".geo", rightValue=pt)).first().getInfo()
    except Exception:
        hit = None
    if not hit:
        box2.value = f"<div style='padding:10px'>{loc.address.split(',')[0]} — no parcel here</div>"
        return
    pan = int(hit["properties"]["PAN"])
    if pan in LOOKUP2:
        show_parcel(pan, LOOKUP2[pan], loc.address.split(",")[0], target=box2)
    else:
        box2.value = f"<div style='padding:10px'>parcel {pan} is not scored on this date</div>"

go2.on_click(do_search2)
F.on_interaction(on_click2)

V2 = valid_dates()
picker2 = DatePicker(value=hot_day.date(), min=V2.min().date(),
                     max=max(V2.max(), pd.Timestamp.today().normalize()).date(),
                     description="date")
picker2.observe(lambda c: c["type"] == "change" and c["new"] and render2(c["new"]), names="value")

render2(hot_day)
display(VBox([hint2, HBox([picker2, search2, go2]), F, status2, box2]))

In [ ]:
# =============================================================================
# Cell 12 — Inside and outside the perimeter, across ten random fires
# =============================================================================
# Two bands per fire, each 300 m deep from the edge: one outside the perimeter,
# one inside. Both are compared against the same day's borough-wide score
# distribution as a percentile rank — 50 is the borough median, 90 the top
# decile. A rank is comparable across fires in a way a raw score is not.
#
# The outer band is uncontaminated by post-fire vegetation but is only a proxy
# for the burn. The inner band is the burn itself, and is contaminated: LANDFIRE
# has already reclassified those pixels toward early seral, so their FUEL and
# therefore their score are depressed. The gap between the two is the measure of
# that contamination — if inside ranks well below outside, the model is reading
# current vegetation rather than the conditions that carried the fire.
#
# Rings with few parcels are flagged and excluded from the summary.
# =============================================================================
import random

random.seed(7)
N_FIRES = 10
MIN_N = 8          # bands smaller than this are reported but not summarised
DEPTH = 300        # metres, both directions

_all = [r for r in _fire_rows if r[1] is not None]
pool = [r for r in _all
        if FWI.index.min() <= r[1].normalize() <= FWI.index.max()]
print(f"{len(_all) - len(pool)} fires dropped: outside the "
      f"{FWI.index.min().date()}–{FWI.index.max().date()} FWI range")
assert len(pool) >= 2, "need at least two dated perimeters in range"

picks = random.sample(pool, min(N_FIRES, len(pool)))
print(f"{len(picks)} fires sampled from {len(pool)}\n")

# Score each distinct date once, not once per fire.
_day_cache = {}
def scored(day):
    day = pd.Timestamp(day)
    if day not in _day_cache:
        ensure_fwi(day)
        _day_cache[day] = score(day)
    return _day_cache[day]

def rank_in(df, mean):
    """Percentile rank of `mean` within this day's borough distribution."""
    return round(float((df["RISK"] < mean).mean()) * 100, 1)

def band_stats(df, band):
    pans = [int(x["properties"]["PAN"]) for x in
            PARCELS_FC.filterBounds(band).limit(20000).getInfo()["features"]]
    sub = df[df["PAN"].isin(pans)]
    if len(sub) == 0:
        return 0, float("nan"), float("nan")
    m = sub["RISK"].mean()
    return len(sub), round(m, 1), rank_in(df, m)

rows = []
for f, ts, approx in picks:
    hot_day = ts.normalize()
    cool_day = hot_day + pd.Timedelta(days=21)

    g2m = ee.Feature(f["geometry"]).geometry().transform(WORK_CRS, 1)
    outer = g2m.buffer(DEPTH, 1).difference(g2m, 1)

    # Negative buffer collapses on small or narrow fires; check before using it.
    inner_core = g2m.buffer(-DEPTH, 1)
    try:
        usable = inner_core.area(1).getInfo() > 0
    except Exception:
        usable = False
    inner = (inner_core.difference(g2m.buffer(-2 * DEPTH, 1), 1)
             if usable else None)

    hot = scored(hot_day)
    cool = scored(cool_day)
    if hot is None or cool is None:
        continue

    n_out, mean_out, rk_out = band_stats(hot, outer)
    if n_out == 0:
        continue

    n_in, mean_in, rk_in = (band_stats(hot, inner) if inner is not None
                            else (0, float("nan"), float("nan")))

    csub = cool[cool["PAN"].isin(
        [int(x["properties"]["PAN"]) for x in
         PARCELS_FC.filterBounds(outer).limit(20000).getInfo()["features"]])]

    rows.append({
        "fire":   (f["properties"].get("NAME")
                   or f["properties"].get("Incid_Name") or "Unnamed"),
        "date":   hot_day.date(),
        "approx": approx,
        "n_out":  n_out,
        "rank_out": rk_out,
        "n_in":   n_in,
        "rank_in": rk_in,
        "gap":    round(rk_in - rk_out, 1) if n_in else float("nan"),
        "cool_drop": round(mean_out - csub["RISK"].mean(), 1) if len(csub) else float("nan"),
    })

res = pd.DataFrame(rows).sort_values("rank_out", ascending=False)
display(res)

print(f"\n{len(res)} fires measured")
out_ok = res[res["n_out"] >= MIN_N]
in_ok = res[(res["n_in"] >= MIN_N) & res["gap"].notna()]

print(f"  outer bands usable (>= {MIN_N} parcels): {len(out_ok)} of {len(res)}")
print(f"  inner bands usable (>= {MIN_N} parcels): {len(in_ok)} of {len(res)}")

if len(out_ok):
    print(f"\nOUTER  median rank: {out_ok['rank_out'].median():.0f}"
          f"   above borough median: {int((out_ok['rank_out'] > 50).sum())}/{len(out_ok)}"
          f"   top decile: {int((out_ok['rank_out'] > 90).sum())}/{len(out_ok)}")
if len(in_ok):
    print(f"INNER  median rank: {in_ok['rank_in'].median():.0f}"
          f"   above borough median: {int((in_ok['rank_in'] > 50).sum())}/{len(in_ok)}"
          f"   top decile: {int((in_ok['rank_in'] > 90).sum())}/{len(in_ok)}")
    print(f"INNER minus OUTER, median gap: {in_ok['gap'].median():+.0f} rank points"
          f"   ({int((in_ok['gap'] < 0).sum())} of {len(in_ok)} fires rank lower inside)")